# Module 2: Feature Engineering
**Customer Intelligence Platform**

Goal: turn the cleaned transaction-level data into a **customer-level feature table** (RFM + extended features), ready to feed into K-Means in Module 3.

## 1: Apply Module 1 Cleaning Decisions

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('../data/raw/online_retail_II.csv')

In [2]:
# Apply the 3 cleaning decisions documented in Module 1
df = df.dropna(subset=['Customer ID'])
df = df[~df['Invoice'].astype(str).str.startswith('C')]
df = df[(df['Quantity'] > 0) & (df['Price'] > 0)]  

In [3]:
# Shape after applying the 3 decisions from Module 1
print(df.shape)
print(df.isna().sum()) 

(805549, 8)
Invoice        0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
Price          0
Customer ID    0
Country        0
dtype: int64


## 2: Create TotalPrice Column

In [4]:
df['TotalPrice'] = df['Quantity'] * df['Price']

## 3: Calculate Recency

In [ ]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
reference_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

recency_df = (df.groupby('Customer ID')['InvoiceDate'].max()
              .apply(lambda x: (reference_date - x).days)
              .reset_index(name='Recency'))

## 4 Calculate Frequency 

In [6]:
frequency_df = (df.groupby('Customer ID')['Invoice'].nunique()
                 .reset_index(name='Frequency'))

## 5 calculate Monetary

In [7]:
monetary_df = (df.groupby('Customer ID')['TotalPrice'].sum()
                .reset_index(name='Monetary'))

## 6 Combine into rfm table 

In [8]:
rfm = recency_df.merge(frequency_df, on='Customer ID').merge(monetary_df, on='Customer ID')
print(rfm.shape)
rfm.head()

(5878, 4)


,Customer ID,Recency,Frequency,Monetary
0,12346.0,326,12,77556.46
1,12347.0,2,8,5633.32
2,12348.0,75,5,2019.40
3,12349.0,19,4,4428.69
4,12350.0,310,1,334.40


## Step 7: Engineer Extended Features

Beyond basic RFM, we add:
- **Tenure** — days between a customer's first and last purchase
- **AOV (Average Order Value)** — Monetary / Frequency
- **Purchase Variability** — std deviation of days between a customer's consecutive orders (irregular vs. regular buying pattern)
- **Product Diversity** — number of unique products (StockCode) a customer has purchased (the dataset has no explicit product-category column, so unique-product count is used as the diversity signal)

In [9]:
# Tenure
customer_dates = (df.groupby('Customer ID')['InvoiceDate'].agg(['min', 'max']).reset_index())
customer_dates['Tenure'] = (customer_dates['max'] - customer_dates['min']).dt.days

rfm = rfm.merge(customer_dates[['Customer ID', 'Tenure']], on='Customer ID', how='left')

In [10]:
# Average Order Value
rfm['AOV'] = rfm['Monetary'] / rfm['Frequency']

In [11]:
# Purchase frequency variability: std of days between consecutive invoices per customer
invoice_dates = df.groupby(['Customer ID', 'Invoice'])['InvoiceDate'].first().reset_index()
invoice_dates = invoice_dates.sort_values(['Customer ID', 'InvoiceDate'])
invoice_dates['days_since_prev'] = invoice_dates.groupby('Customer ID')['InvoiceDate'].diff().dt.days

purchase_variability = (invoice_dates.groupby('Customer ID')['days_since_prev']
                         .std().reset_index(name='Purchase_Variability'))
# Customers with only 1 order have no gap to measure -> fill with 0 (no observed variability)
purchase_variability['Purchase_Variability'] = purchase_variability['Purchase_Variability'].fillna(0)

rfm = rfm.merge(purchase_variability, on='Customer ID', how='left')

In [12]:
# Product diversity
product_diversity = df.groupby('Customer ID')['StockCode'].nunique().reset_index(name='Product_Diversity')
rfm = rfm.merge(product_diversity, on='Customer ID', how='left')

print(rfm.shape)
rfm.head()

(5878, 8)


,Customer ID,Recency,Frequency,Monetary,Tenure,AOV,Purchase_Variability,Product_Diversity
0,12346.0,326,12,77556.46,400,6463.038333,65.426989,27
1,12347.0,2,8,5633.32,402,704.165000,19.035055,126
2,12348.0,75,5,2019.40,362,403.880000,57.703264,25
3,12349.0,19,4,4428.69,570,1107.172500,187.040994,138
4,12350.0,310,1,334.40,0,334.400000,0.000000,17


## Step 8: Handle Skewness, Scale & Save

In [13]:
# Skew check BEFORE transform
base_cols = ['Recency', 'Frequency', 'Monetary', 'Tenure', 'AOV', 'Purchase_Variability', 'Product_Diversity']
rfm[base_cols].skew()

Recency                  0.887198
Frequency               12.639951
Monetary                25.314235
Tenure                   0.388825
AOV                     57.008610
Purchase_Variability     2.331570
Product_Diversity        6.224360
dtype: float64

**Skew observed:** `Recency` (~0.89) and `Tenure` (~0.39) are only mildly skewed, so they're left as-is. `Frequency`, `Monetary`, `AOV`, `Purchase_Variability`, and `Product_Diversity` are all heavily right-skewed (several well above the common >1 rule-of-thumb threshold), so a `log1p` transform is applied to each of those.

In [14]:
# Log-transform the skewed features
rfm['Frequency_log'] = np.log1p(rfm['Frequency'])
rfm['Monetary_log'] = np.log1p(rfm['Monetary'])
rfm['AOV_log'] = np.log1p(rfm['AOV'])
rfm['Purchase_Variability_log'] = np.log1p(rfm['Purchase_Variability'])
rfm['Product_Diversity_log'] = np.log1p(rfm['Product_Diversity'])

log_cols = ['Frequency_log', 'Monetary_log', 'AOV_log', 'Purchase_Variability_log', 'Product_Diversity_log']
rfm[log_cols].skew()

Frequency_log               1.001471
Monetary_log                0.265113
AOV_log                     0.042984
Purchase_Variability_log    0.049124
Product_Diversity_log      -0.281116
dtype: float64

In [15]:
# Scale the final feature set
features = ['Recency', 'Frequency_log', 'Monetary_log', 'Tenure',
            'AOV_log', 'Purchase_Variability_log', 'Product_Diversity_log']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(rfm[features])

scaled_features = pd.DataFrame(X_scaled, columns=features, index=rfm.index)
scaled_features = pd.concat([rfm[['Customer ID']], scaled_features], axis=1)

scaled_features.head()

,Customer ID,Recency,Frequency_log,Monetary_log,Tenure,AOV_log,Purchase_Variability_log,Product_Diversity_log
0,12346.0,0.595584,1.254496,3.186625,0.490667,4.264242,0.982440,-0.350753
1,12347.0,-0.952279,0.800166,1.297127,0.498395,1.253001,0.397709,0.889667
2,12348.0,-0.603532,0.299207,0.558100,0.343827,0.498904,0.922140,-0.411550
3,12349.0,-0.871064,0.073946,1.123790,1.147582,1.867373,1.490062,0.963738
4,12350.0,0.519146,-1.058146,-0.735888,-1.055014,0.243026,-1.064573,-0.713229


In [16]:
# Save both the full engineered feature table and the scaled version for Module 3
rfm.to_csv('../data/processed/customer_features.csv', index=False)
scaled_features.to_csv('../data/processed/customer_features_scaled.csv', index=False)

print('Saved customer_features.csv and customer_features_scaled.csv to data/processed/')

Saved customer_features.csv and customer_features_scaled.csv to data/processed/


### Module 2 Summary

- Started from **805,549** cleaned transaction rows -> aggregated down to **5,878 unique customers**
- Built core RFM: `Recency`, `Frequency`, `Monetary`
- Added extended features: `Tenure`, `AOV`, `Purchase_Variability`, `Product_Diversity`
- Log-transformed 5 heavily skewed features, scaled all 7 final features with `StandardScaler`
- Saved two files to `data/processed/`:
  - `customer_features.csv` — full engineered table (raw + log columns, for reference/interpretation)
  - `customer_features_scaled.csv` — final scaled feature set, ready to feed directly into K-Means (Module 3)